# Notebook 08 v3 — Select a Train-Only Abstention Policy

The frozen Stage 07 model remains unchanged:

- model: XGBoost;
- feature set: `I_full_40`;
- selected trial: 10;
- raw model features: 40.

This notebook changes only the downstream decision policy. It searches a small,
preregistered train-OOF grid:

```text
Breadth regime gate
+ fixed raw-score threshold
+ daily Top-5% maximum cap
+ minimum signals per date = 0
```

The 2021–2024 evaluation period is not opened. Economic outcomes and portfolio
metrics are not used. The selected policy may abstain completely on a date.


In [1]:
from __future__ import annotations

from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import yaml

print("Python:", sys.version.split()[0])
print("numpy:", np.__version__)
print("pandas:", pd.__version__)


Python: 3.12.2
numpy: 1.26.4
pandas: 2.2.3


## 1. Locate the repository and import frozen implementations


In [2]:
def locate_repository_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (
            (candidate / "configs").exists()
            and (candidate / "notebooks").exists()
            and (candidate / "src").exists()
        ):
            return candidate
    raise FileNotFoundError(
        "Repository root was not found. Open the project root in VS Code."
    )


REPOSITORY_ROOT = locate_repository_root(Path.cwd())
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from src.models.abstention_policy import (
    ABSTENTION_POLICY_SCHEMA_VERSION,
    AbstentionPolicy,
    apply_abstention_policy,
    load_candidate_regime_sources,
    select_abstention_policy,
    summarize_abstention_policy,
    threshold_grid_from_baseline_scores,
)
from src.models.policy_selection import (
    DailyTopFractionPolicy,
    apply_daily_top_fraction_policy,
    summarize_policy_predictions,
)
from src.utils.paths import repository_result_paths
from src.utils.reproducibility import (
    git_commit_sha,
    software_manifest,
    stable_object_hash,
)

print("Repository root:", REPOSITORY_ROOT)


Repository root: E:\Iran_stock_trade\financial-ml-decision-support


e:\Iran_stock_trade\financial-ml-decision-support\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load frozen model lineage and train-only policy configuration


In [3]:
def load_yaml(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file_obj:
        value = yaml.safe_load(file_obj)
    if not isinstance(value, dict):
        raise ValueError(f"Configuration must be a mapping: {path}")
    return value


def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file_obj:
        value = json.load(file_obj)
    if not isinstance(value, dict):
        raise ValueError(f"JSON artifact must be an object: {path}")
    return value


paths_config = load_yaml(
    REPOSITORY_ROOT / "configs" / "paths.yaml"
)
policy_config = load_yaml(
    REPOSITORY_ROOT / "configs" / "signal_policy.yaml"
)
RESULT_PATHS = repository_result_paths(
    REPOSITORY_ROOT,
    paths_config,
)

prediction_path = (
    REPOSITORY_ROOT
    / policy_config["input"]["prediction_file"]
)
candidate_root = (
    REPOSITORY_ROOT
    / policy_config["input"]["candidate_directory"]
)
candidate_glob = str(
    policy_config["input"]["candidate_file_glob"]
)

stage06_decision_path = (
    RESULT_PATHS["manifests"]
    / "06_model_selection_decision.json"
)
stage06_manifest_path = (
    RESULT_PATHS["manifests"]
    / "06_optuna_model_selection_manifest.json"
)
stage07_manifest_path = (
    RESULT_PATHS["manifests"]
    / "07_frozen_model_training_manifest.json"
)

for required_path in [
    prediction_path,
    candidate_root,
    stage06_decision_path,
    stage06_manifest_path,
    stage07_manifest_path,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required frozen input is missing: {required_path}"
        )

stage06_decision = load_json(stage06_decision_path)
stage06_manifest = load_json(stage06_manifest_path)
stage07_manifest = load_json(stage07_manifest_path)

SELECTED_MODEL = str(
    stage06_decision["primary_selected_model"]
)
SELECTED_FEATURE_SET = str(
    stage06_decision["primary_selected_feature_set"]
)
SELECTED_FEATURES = list(
    stage06_decision["primary_selected_features"]
)
SELECTED_TRIAL = int(
    stage06_decision["selected_trial_numbers"][
        SELECTED_MODEL
    ]
)

assert policy_config["status"] == (
    "stage_08_configured_v3_train_only_abstention_policy_selection"
)
assert ABSTENTION_POLICY_SCHEMA_VERSION == (
    policy_config["abstention_policy"][
        "implementation_schema_version"
    ]
)
assert stage06_manifest["unseen_test_used"] is False
assert stage06_decision["unseen_test_used"] is False
assert stage07_manifest["unseen_test_used"] is False

assert SELECTED_MODEL == "xgboost"
assert SELECTED_FEATURE_SET == "I_full_40"
assert SELECTED_TRIAL == 10
assert len(SELECTED_FEATURES) == 40
assert stage07_manifest["model"]["selected_model"] == SELECTED_MODEL
assert (
    stage07_manifest["model"]["selected_feature_set"]
    == SELECTED_FEATURE_SET
)
assert int(
    stage07_manifest["model"]["selected_trial_number"]
) == SELECTED_TRIAL
assert list(
    stage07_manifest["features"]["model_features"]
) == SELECTED_FEATURES

print("Selected model:", SELECTED_MODEL)
print("Selected feature set:", SELECTED_FEATURE_SET)
print("Selected trial:", SELECTED_TRIAL)
print("Raw model features:", len(SELECTED_FEATURES))
print(
    "Frozen model SHA256:",
    stage07_manifest["model"]["model_sha256"],
)


Selected model: xgboost
Selected feature set: I_full_40
Selected trial: 10
Raw model features: 40
Frozen model SHA256: 73a781551cdabd6bb67f5b9c0836a8683372e46487d9aade29e6320006086c1f


## 3. Load exact selected-model OOF rows and join Breadth regime


In [4]:
prediction_columns = [
    "model_name",
    "feature_set_name",
    "fold_id",
    "event_id",
    "symbol",
    "dEven",
    "event_end_date",
    "meta_label",
    "probability_positive",
    "validation_metric_weight",
]
all_oof = pd.read_csv(
    prediction_path,
    usecols=prediction_columns,
    low_memory=False,
)
oof = all_oof.loc[
    all_oof["model_name"].eq(SELECTED_MODEL)
    & all_oof["feature_set_name"].eq(
        SELECTED_FEATURE_SET
    )
].copy()

oof["fold_id"] = pd.to_numeric(
    oof["fold_id"],
    errors="raise",
).astype(int)
oof["dEven"] = pd.to_datetime(
    oof["dEven"],
    errors="raise",
).dt.normalize()
oof["event_end_date"] = pd.to_datetime(
    oof["event_end_date"],
    errors="raise",
).dt.normalize()
oof["meta_label"] = pd.to_numeric(
    oof["meta_label"],
    errors="raise",
).astype(int)
oof["probability_positive"] = pd.to_numeric(
    oof["probability_positive"],
    errors="raise",
)

candidate_paths = sorted(
    candidate_root.glob(candidate_glob)
)
if len(candidate_paths) != 499:
    raise AssertionError(
        f"Expected 499 train candidate files, observed "
        f"{len(candidate_paths)}."
    )

candidate_regimes, candidate_source_errors_df = (
    load_candidate_regime_sources(
        candidate_paths
    )
)
candidate_source_errors_df.to_csv(
    RESULT_PATHS["audits"]
    / "08_abstention_candidate_source_errors.csv",
    index=False,
    encoding="utf-8-sig",
)
if not candidate_source_errors_df.empty:
    display(candidate_source_errors_df.head(20))
    raise RuntimeError(
        "Candidate regime source errors exist."
    )

# Stage 06 did not read event_id from candidate CSV files. It reconstructed
# event_id as symbol::YYYY-MM-DD using the symbol encoded in each filename.
# The helper above reproduces that exact frozen identity rule.
assert candidate_regimes["event_id"].is_unique
assert candidate_regimes[
    "market_breadth_regime"
].notna().all()

oof["event_id"] = oof["event_id"].astype(str)
oof_event_ids = set(oof["event_id"])
candidate_event_ids = set(
    candidate_regimes["event_id"]
)
missing_regime_event_ids = sorted(
    oof_event_ids - candidate_event_ids
)
assert not missing_regime_event_ids

oof = oof.merge(
    candidate_regimes[
        [
            "event_id",
            "market_breadth_regime",
        ]
    ],
    on="event_id",
    how="left",
    validate="one_to_one",
)
oof = oof.sort_values(
    ["fold_id", "dEven", "symbol", "event_id"],
    kind="stable",
).reset_index(drop=True)

expected_rows = int(
    policy_config["input"]["expected_oof_rows"]
)
expected_fold_counts = {
    int(key): int(value)
    for key, value in policy_config[
        "input"
    ]["expected_fold_counts"].items()
}
observed_fold_counts = (
    oof.groupby("fold_id", sort=True)
    .size()
    .astype(int)
    .to_dict()
)

assert len(oof) == expected_rows
assert observed_fold_counts == expected_fold_counts
assert oof["event_id"].is_unique
assert oof["meta_label"].isin([0, 1]).all()
assert np.isfinite(
    oof["probability_positive"].to_numpy(dtype=float)
).all()
assert oof["probability_positive"].between(
    0.0,
    1.0,
).all()
assert oof["market_breadth_regime"].notna().all()

oof_input_audit_df = (
    oof.groupby("fold_id", sort=True)
    .agg(
        events=("event_id", "size"),
        dates=("dEven", "nunique"),
        first_date=("dEven", "min"),
        last_date=("dEven", "max"),
        positive_fraction=("meta_label", "mean"),
        minimum_raw_score=(
            "probability_positive",
            "min",
        ),
        mean_raw_score=(
            "probability_positive",
            "mean",
        ),
        maximum_raw_score=(
            "probability_positive",
            "max",
        ),
        regimes=(
            "market_breadth_regime",
            "nunique",
        ),
    )
    .reset_index()
)
oof_input_audit_df["selected_model"] = SELECTED_MODEL
oof_input_audit_df[
    "selected_feature_set"
] = SELECTED_FEATURE_SET
oof_input_audit_df["selected_trial"] = SELECTED_TRIAL
oof_input_audit_df[
    "candidate_regime_source_rows"
] = int(len(candidate_regimes))
oof_input_audit_df[
    "missing_oof_regime_event_ids"
] = int(len(missing_regime_event_ids))
oof_input_audit_df[
    "event_id_reconstruction_rule"
] = "symbol::YYYY-MM-DD_from_candidate_filename_and_dEven"
oof_input_audit_df["unseen_test_used"] = False
oof_input_audit_df.to_csv(
    RESULT_PATHS["audits"]
    / "08_abstention_oof_input_audit.csv",
    index=False,
    encoding="utf-8-sig",
)
display(oof_input_audit_df)


,fold_id,events,dates,first_date,last_date,positive_fraction,minimum_raw_score,mean_raw_score,maximum_raw_score,regimes,selected_model,selected_feature_set,selected_trial,candidate_regime_source_rows,missing_oof_regime_event_ids,event_id_reconstruction_rule,unseen_test_used
0,1,10344,92,2017-09-19,2018-01-31,0.385731,0.297799,0.542050,0.722038,2,xgboost,I_full_40,10,118464,0,symbol::YYYY-MM-DD_from_candidate_filename_and...,False
1,2,10408,91,2018-02-03,2018-06-27,0.466660,0.308411,0.526818,0.732247,3,xgboost,I_full_40,10,118464,0,symbol::YYYY-MM-DD_from_candidate_filename_and...,False
2,3,10397,136,2018-06-30,2019-01-15,0.726363,0.273232,0.461357,0.750240,4,xgboost,I_full_40,10,118464,0,symbol::YYYY-MM-DD_from_candidate_filename_and...,False
3,4,10319,231,2019-01-16,2020-01-04,0.888071,0.303482,0.529864,0.834761,3,xgboost,I_full_40,10,118464,0,symbol::YYYY-MM-DD_from_candidate_filename_and...,False
4,5,10372,288,2020-01-05,2021-03-14,0.507135,0.329853,0.588739,0.830716,5,xgboost,I_full_40,10,118464,0,symbol::YYYY-MM-DD_from_candidate_filename_and...,False


## 4. Reconstruct the frozen old policy as a train-OOF baseline


In [5]:
baseline_config = policy_config["baseline_policy"]
baseline_policy = DailyTopFractionPolicy(
    fraction=float(
        baseline_config["selected_fraction"]
    ),
    minimum_signals_per_date=int(
        baseline_config[
            "minimum_signals_per_date"
        ]
    ),
)

baseline_predictions = (
    apply_daily_top_fraction_policy(
        oof,
        policy=baseline_policy,
    )
)
baseline_summary, baseline_fold_metrics = (
    summarize_policy_predictions(
        baseline_predictions,
        policy=baseline_policy,
    )
)

expected_baseline_signals = int(
    policy_config["input"][
        "expected_baseline_signals"
    ]
)
expected_baseline_fold_signals = {
    int(key): int(value)
    for key, value in policy_config[
        "input"
    ][
        "expected_baseline_fold_signals"
    ].items()
}
observed_baseline_fold_signals = (
    baseline_fold_metrics.set_index("fold_id")[
        "signals"
    ].astype(int).to_dict()
)

assert int(
    baseline_summary["signals"]
) == expected_baseline_signals
assert (
    observed_baseline_fold_signals
    == expected_baseline_fold_signals
)
assert int(
    baseline_summary["false_positive"]
) == int(
    stage06_decision["selected_false_positive"]
)
assert int(
    baseline_summary["true_positive"]
) == int(
    stage06_decision["selected_true_positive"]
)
assert np.isclose(
    float(baseline_summary["precision"]),
    float(stage06_decision["selected_precision"]),
)

baseline_summary_df = pd.DataFrame([{
    "policy_name": "old_daily_top5_minimum_one",
    **{
        key: value
        for key, value in baseline_summary.items()
        if key != "policy"
    },
}])
baseline_summary_df.to_csv(
    RESULT_PATHS["audits"]
    / "08_abstention_baseline_policy_summary.csv",
    index=False,
    encoding="utf-8-sig",
)
baseline_fold_metrics.to_csv(
    RESULT_PATHS["audits"]
    / "08_abstention_baseline_fold_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Baseline signals:", baseline_summary["signals"])
print(
    "Baseline false positives:",
    baseline_summary["false_positive"],
)
print("Baseline precision:", baseline_summary["precision"])


Baseline signals: 3016
Baseline false positives: 1029
Baseline precision: 0.6588196286472149


## 5. Evaluate the preregistered abstention-policy grid


In [6]:
abstention_config = policy_config[
    "abstention_policy"
]
coverage_config = policy_config[
    "coverage_constraints"
]

threshold_grid = threshold_grid_from_baseline_scores(
    baseline_predictions.loc[
        baseline_predictions["selected_signal"],
        "probability_positive",
    ],
    quantiles=abstention_config[
        "threshold_quantiles"
    ],
)

minimum_pooled_coverage = float(
    coverage_config[
        "minimum_pooled_signal_coverage_vs_baseline"
    ]
)
minimum_fold_coverage = float(
    coverage_config[
        "minimum_each_fold_signal_coverage_vs_baseline"
    ]
)
maximum_daily_fraction = float(
    abstention_config[
        "maximum_daily_fraction"
    ]
)
minimum_signals_per_date = int(
    abstention_config[
        "minimum_signals_per_date"
    ]
)
assert minimum_signals_per_date == 0

candidate_summary_rows = []
candidate_fold_parts = []
candidate_prediction_map = {}

for gate_name, gate_spec in abstention_config[
    "regime_gates"
].items():
    allowed_regimes = tuple(
        str(value)
        for value in gate_spec[
            "allowed_regimes"
        ]
    )
    gate_complexity = int(
        gate_spec["gate_complexity"]
    )

    for threshold_row in threshold_grid.itertuples(
        index=False
    ):
        threshold_quantile = float(
            threshold_row.threshold_quantile
        )
        minimum_score = float(
            threshold_row.minimum_score
        )
        candidate_id = (
            f"{gate_name}"
            f"__q{int(round(threshold_quantile * 1000)):04d}"
        )

        policy = AbstentionPolicy(
            gate_name=gate_name,
            allowed_regimes=allowed_regimes,
            minimum_score=minimum_score,
            maximum_daily_fraction=(
                maximum_daily_fraction
            ),
            minimum_signals_per_date=(
                minimum_signals_per_date
            ),
        )
        policy_predictions = apply_abstention_policy(
            oof,
            policy=policy,
        )
        summary, fold_metrics = (
            summarize_abstention_policy(
                policy_predictions,
                policy=policy,
                baseline_signal_count=(
                    expected_baseline_signals
                ),
                baseline_fold_signal_counts=(
                    expected_baseline_fold_signals
                ),
                minimum_pooled_coverage=(
                    minimum_pooled_coverage
                ),
                minimum_fold_coverage=(
                    minimum_fold_coverage
                ),
            )
        )

        candidate_summary_rows.append({
            "candidate_id": candidate_id,
            "gate_name": gate_name,
            "gate_complexity": gate_complexity,
            "allowed_regimes": json.dumps(
                list(allowed_regimes),
                ensure_ascii=False,
            ),
            "threshold_quantile": (
                threshold_quantile
            ),
            "minimum_score": minimum_score,
            **{
                key: value
                for key, value in summary.items()
                if key != "policy"
            },
        })
        fold_metrics = fold_metrics.copy()
        fold_metrics["candidate_id"] = candidate_id
        fold_metrics["gate_name"] = gate_name
        fold_metrics[
            "threshold_quantile"
        ] = threshold_quantile
        fold_metrics[
            "minimum_score"
        ] = minimum_score
        candidate_fold_parts.append(fold_metrics)
        candidate_prediction_map[
            candidate_id
        ] = policy_predictions

candidate_grid_df = pd.DataFrame(
    candidate_summary_rows
)
candidate_fold_metrics_df = pd.concat(
    candidate_fold_parts,
    ignore_index=True,
)
ranked_policy_grid_df = select_abstention_policy(
    candidate_grid_df
)

selected_rows = ranked_policy_grid_df.loc[
    ranked_policy_grid_df[
        "selected_by_policy_hierarchy"
    ]
]
assert len(selected_rows) == 1
selected_candidate_row = selected_rows.iloc[0]
SELECTED_POLICY_ID = str(
    selected_candidate_row["candidate_id"]
)

ranked_policy_grid_df.to_csv(
    RESULT_PATHS["audits"]
    / "08_abstention_policy_candidate_grid.csv",
    index=False,
    encoding="utf-8-sig",
)
candidate_fold_metrics_df.to_csv(
    RESULT_PATHS["audits"]
    / "08_abstention_policy_all_fold_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Policy candidates:", len(ranked_policy_grid_df))
print(
    "Candidates passing coverage:",
    int(
        ranked_policy_grid_df[
            "coverage_constraints_pass"
        ].sum()
    ),
)
print("Selected policy ID:", SELECTED_POLICY_ID)
display(
    ranked_policy_grid_df.head(20)
)


Policy candidates: 20
Candidates passing coverage: 6
Selected policy ID: G3_recovery_only__q0000


,candidate_id,gate_name,gate_complexity,allowed_regimes,threshold_quantile,minimum_score,true_positive,false_positive,true_negative,false_negative,...,minimum_fold_signal_coverage,minimum_fold_precision,mean_fold_precision,std_fold_precision,minimum_fold_specificity,coverage_constraints_pass,minimum_pooled_coverage_required,minimum_fold_coverage_required,policy_selection_rank,selected_by_policy_hierarchy
0,G3_recovery_only__q0000,G3_recovery_only,3,"[""recovery_negative"", ""recovery_positive""]",0.00,0.384463,994,404,20613,29829,...,0.295082,0.440559,0.697806,0.187682,0.974819,True,0.25,0.25,1.0,True
1,G2_block_decline_and_deterioration__q0000,G2_block_decline_and_deterioration,2,"[""recovery_negative"", ""recovery_positive"", ""ne...",0.00,0.384463,1024,417,20600,29799,...,0.344262,0.440559,0.697014,0.186319,0.974819,True,0.25,0.25,2.0,False
2,G1_block_broad_decline__q0250,G1_block_broad_decline,1,"[""recovery_negative"", ""recovery_positive"", ""de...",0.25,0.544538,1325,783,20234,29498,...,0.400673,0.436404,0.647803,0.177464,0.958386,True,0.25,0.25,3.0,False
3,G0_all__q0250,G0_all,0,"[""broad_decline"", ""recovery_negative"", ""recove...",0.25,0.544538,1437,825,20192,29386,...,0.400673,0.436404,0.655753,0.174472,0.951682,True,0.25,0.25,4.0,False
4,G1_block_broad_decline__q0000,G1_block_broad_decline,1,"[""recovery_negative"", ""recovery_positive"", ""de...",0.00,0.384463,1858,987,20030,28965,...,0.745156,0.440285,0.644244,0.173255,0.944589,True,0.25,0.25,5.0,False
5,G0_all__q0000,G0_all,0,"[""broad_decline"", ""recovery_negative"", ""recove...",0.00,0.384463,1987,1029,19988,28836,...,1.000000,0.440285,0.653846,0.169952,0.944589,True,0.25,0.25,6.0,False
6,G3_recovery_only__q0750,G3_recovery_only,3,"[""recovery_negative"", ""recovery_positive""]",0.75,0.651050,248,69,20948,30575,...,0.019435,0.352941,0.664568,0.270755,0.993545,False,0.25,0.25,NaN,False
7,G2_block_decline_and_deterioration__q0750,G2_block_decline_and_deterioration,2,"[""recovery_negative"", ""recovery_positive"", ""ne...",0.75,0.651050,269,77,20940,30554,...,0.019435,0.352941,0.666209,0.271033,0.991980,False,0.25,0.25,NaN,False
8,G3_recovery_only__q0650,G3_recovery_only,3,"[""recovery_negative"", ""recovery_positive""]",0.65,0.632068,334,116,20901,30489,...,0.075758,0.430769,0.702223,0.208523,0.990219,False,0.25,0.25,NaN,False
9,G2_block_decline_and_deterioration__q0650,G2_block_decline_and_deterioration,2,"[""recovery_negative"", ""recovery_positive"", ""ne...",0.65,0.632068,361,124,20893,30462,...,0.075972,0.430769,0.707380,0.208515,0.988654,False,0.25,0.25,NaN,False


## 6. Freeze selected train-only abstention policy


In [7]:
selected_gate_name = str(
    selected_candidate_row["gate_name"]
)
selected_gate_spec = abstention_config[
    "regime_gates"
][selected_gate_name]
selected_allowed_regimes = tuple(
    str(value)
    for value in selected_gate_spec[
        "allowed_regimes"
    ]
)
selected_minimum_score = float(
    selected_candidate_row["minimum_score"]
)
selected_threshold_quantile = float(
    selected_candidate_row[
        "threshold_quantile"
    ]
)

SELECTED_POLICY = AbstentionPolicy(
    gate_name=selected_gate_name,
    allowed_regimes=selected_allowed_regimes,
    minimum_score=selected_minimum_score,
    maximum_daily_fraction=(
        maximum_daily_fraction
    ),
    minimum_signals_per_date=0,
)
selected_predictions = candidate_prediction_map[
    SELECTED_POLICY_ID
].copy()
selected_summary, selected_fold_metrics = (
    summarize_abstention_policy(
        selected_predictions,
        policy=SELECTED_POLICY,
        baseline_signal_count=(
            expected_baseline_signals
        ),
        baseline_fold_signal_counts=(
            expected_baseline_fold_signals
        ),
        minimum_pooled_coverage=(
            minimum_pooled_coverage
        ),
        minimum_fold_coverage=(
            minimum_fold_coverage
        ),
    )
)

selected_predictions.to_csv(
    RESULT_PATHS["predictions"]
    / "08_abstention_oof_policy_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)
selected_fold_metrics.to_csv(
    RESULT_PATHS["audits"]
    / "08_abstention_selected_policy_fold_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

selected_date_audit_df = (
    selected_predictions.groupby(
        "dEven",
        sort=True,
        observed=False,
    )
    .agg(
        fold_id=("fold_id", "first"),
        market_breadth_regime=(
            "market_breadth_regime",
            "first",
        ),
        candidate_events=(
            "event_id",
            "size",
        ),
        gate_pass_events=(
            "market_gate_pass",
            "sum",
        ),
        threshold_pass_events=(
            "score_threshold_pass",
            "sum",
        ),
        policy_eligible_events=(
            "policy_eligible",
            "sum",
        ),
        daily_maximum_quota=(
            "daily_maximum_quota",
            "first",
        ),
        selected_signals=(
            "selected_signal",
            "sum",
        ),
        selected_score_cutoff=(
            "daily_selected_score_cutoff",
            "first",
        ),
    )
    .reset_index()
)
selected_date_audit_df[
    "zero_signal_date"
] = selected_date_audit_df[
    "selected_signals"
].eq(0)
selected_date_audit_df.to_csv(
    RESULT_PATHS["audits"]
    / "08_abstention_selected_policy_date_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

comparison_df = pd.DataFrame([
    {
        "policy_name": "old_daily_top5_minimum_one",
        "signals": int(
            baseline_summary["signals"]
        ),
        "true_positive": int(
            baseline_summary["true_positive"]
        ),
        "false_positive": int(
            baseline_summary["false_positive"]
        ),
        "precision": float(
            baseline_summary["precision"]
        ),
        "specificity": float(
            baseline_summary["specificity"]
        ),
        "sensitivity": float(
            baseline_summary["sensitivity"]
        ),
        "zero_signal_dates": 0,
        "signal_coverage_vs_baseline": 1.0,
    },
    {
        "policy_name": "selected_abstention_policy",
        "signals": int(
            selected_summary["signals"]
        ),
        "true_positive": int(
            selected_summary["true_positive"]
        ),
        "false_positive": int(
            selected_summary["false_positive"]
        ),
        "precision": float(
            selected_summary["precision"]
        ),
        "specificity": float(
            selected_summary["specificity"]
        ),
        "sensitivity": float(
            selected_summary["sensitivity"]
        ),
        "zero_signal_dates": int(
            selected_summary["zero_signal_dates"]
        ),
        "signal_coverage_vs_baseline": float(
            selected_summary["signal_coverage"]
        ),
    },
])
comparison_df.to_csv(
    RESULT_PATHS["audits"]
    / "08_abstention_baseline_vs_selected.csv",
    index=False,
    encoding="utf-8-sig",
)

assert bool(
    selected_summary[
        "coverage_constraints_pass"
    ]
)
assert int(
    selected_summary["signals"]
) >= int(
    np.ceil(
        expected_baseline_signals
        * minimum_pooled_coverage
    )
)
assert selected_fold_metrics[
    "signal_coverage"
].ge(minimum_fold_coverage).all()
assert selected_predictions.groupby(
    "dEven",
    observed=False,
)["selected_signal"].sum().le(
    selected_predictions.groupby(
        "dEven",
        observed=False,
    )["daily_maximum_quota"].first()
).all()

display(comparison_df)
display(selected_fold_metrics)


,policy_name,signals,true_positive,false_positive,precision,specificity,sensitivity,zero_signal_dates,signal_coverage_vs_baseline
0,old_daily_top5_minimum_one,3016,1987,1029,0.658820,0.951040,0.064465,0,1.000000
1,selected_abstention_policy,1398,994,404,0.711016,0.980777,0.032249,452,0.463528


,fold_id,true_positive,false_positive,true_negative,false_negative,signals,baseline_signals,signal_coverage,events,dates,dates_with_signal,precision,specificity,sensitivity
0,1,126,160,6194,3864,286,561,0.509804,10344,92,47,0.440559,0.974819,0.031579
1,2,149,123,5428,4708,272,566,0.480565,10408,91,45,0.547794,0.977842,0.030677
2,3,256,43,2802,7296,299,594,0.503367,10397,136,64,0.856187,0.984886,0.033898
3,4,325,18,1137,8839,343,624,0.549679,10319,231,136,0.947522,0.984416,0.035465
4,5,138,60,5052,5122,198,671,0.295082,10372,288,94,0.696970,0.988263,0.026236


## 7. Write frozen policy and manifest


In [8]:
selected_policy_artifact = {
    "stage": 8,
    "status": "frozen_train_only_abstention_policy",
    "policy_schema_version": (
        ABSTENTION_POLICY_SCHEMA_VERSION
    ),
    "source_model": SELECTED_MODEL,
    "source_feature_set": SELECTED_FEATURE_SET,
    "source_trial": SELECTED_TRIAL,
    "source_model_sha256": stage07_manifest[
        "model"
    ]["model_sha256"],
    "score_policy": {
        "score_column": "probability_positive",
        "interpretation": (
            "uncalibrated_ranking_score"
        ),
        "calibrator_fitted": False,
        "literal_probability_interpretation_allowed": False,
        "threshold_is_operational_cutoff_not_calibrated_probability": True,
    },
    "market_gate": {
        "gate_name": selected_gate_name,
        "allowed_regimes": list(
            selected_allowed_regimes
        ),
        "blocked_regimes": sorted(
            set(
                value
                for gate_spec in abstention_config[
                    "regime_gates"
                ].values()
                for value in gate_spec[
                    "allowed_regimes"
                ]
            )
            - set(selected_allowed_regimes)
        ),
    },
    "score_threshold": {
        "minimum_raw_score": (
            selected_minimum_score
        ),
        "source_quantile_of_baseline_selected_oof_scores": (
            selected_threshold_quantile
        ),
    },
    "daily_cap": {
        "maximum_fraction": (
            maximum_daily_fraction
        ),
        "minimum_signals_per_date": 0,
        "quota_rounding": "ceil",
        "zero_signal_dates_allowed": True,
        "ranking_order": [
            "score_descending",
            "symbol_ascending",
            "event_id_ascending",
        ],
    },
    "coverage_constraints": {
        "minimum_pooled_signal_coverage_vs_baseline": (
            minimum_pooled_coverage
        ),
        "minimum_each_fold_signal_coverage_vs_baseline": (
            minimum_fold_coverage
        ),
        "passed": True,
    },
    "train_oof_selection_metrics": {
        key: value
        for key, value in selected_summary.items()
        if key != "policy"
    },
    "baseline_train_oof_metrics": {
        key: value
        for key, value in baseline_summary.items()
        if key != "policy"
    },
    "selection_hierarchy": policy_config[
        "selection_hierarchy"
    ],
    "safeguards": {
        "unseen_test_loaded": False,
        "unseen_test_labels_used": False,
        "economic_returns_used_for_selection": False,
        "portfolio_metrics_used_for_selection": False,
        "model_retrained": False,
        "hyperparameters_changed": False,
        "feature_set_changed": False,
        "calibration_fitted": False,
        "hard_started_filter_applied": False,
        "minimum_signals_per_date_is_zero": True,
    },
}

selected_policy_artifact[
    "configuration_hash"
] = stable_object_hash({
    "policy_config": policy_config,
    "selected_policy": {
        "gate_name": selected_gate_name,
        "allowed_regimes": list(
            selected_allowed_regimes
        ),
        "minimum_score": (
            selected_minimum_score
        ),
        "threshold_quantile": (
            selected_threshold_quantile
        ),
        "maximum_daily_fraction": (
            maximum_daily_fraction
        ),
        "minimum_signals_per_date": 0,
    },
    "source_model_sha256": stage07_manifest[
        "model"
    ]["model_sha256"],
})

policy_output_path = (
    RESULT_PATHS["manifests"]
    / "08_abstention_signal_policy.json"
)
policy_output_path.write_text(
    json.dumps(
        selected_policy_artifact,
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

manifest = {
    "stage": 8,
    "status": "frozen_after_integrity_checks",
    "stage_version": (
        "v3_train_only_abstention_policy"
    ),
    "notebook": (
        "08_unseen_test_evaluation.ipynb"
    ),
    "git_commit_sha": git_commit_sha(
        REPOSITORY_ROOT
    ),
    "software": software_manifest(),
    "selected_policy_file": str(
        policy_output_path
    ),
    "selected_policy_configuration_hash": (
        selected_policy_artifact[
            "configuration_hash"
        ]
    ),
    "selected_policy_id": SELECTED_POLICY_ID,
    "candidate_policies_evaluated": int(
        len(ranked_policy_grid_df)
    ),
    "candidate_policies_passing_coverage": int(
        ranked_policy_grid_df[
            "coverage_constraints_pass"
        ].sum()
    ),
    "selected_model": SELECTED_MODEL,
    "selected_feature_set": (
        SELECTED_FEATURE_SET
    ),
    "selected_trial": SELECTED_TRIAL,
    "source_model_sha256": stage07_manifest[
        "model"
    ]["model_sha256"],
    "oof_rows": int(len(oof)),
    "oof_folds": sorted(
        oof["fold_id"].unique().astype(int).tolist()
    ),
    "selected_policy": (
        selected_policy_artifact
    ),
    "unseen_test_used": False,
    "economic_returns_used": False,
    "portfolio_metrics_used": False,
}

manifest["configuration_hash"] = (
    stable_object_hash({
        "manifest_without_hash": manifest,
    })
)

manifest_path = (
    RESULT_PATHS["manifests"]
    / "08_abstention_policy_manifest.json"
)
manifest_path.write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print("Policy artifact:", policy_output_path)
print("Policy manifest:", manifest_path)
print(
    "Policy configuration SHA:",
    selected_policy_artifact[
        "configuration_hash"
    ],
)


Policy artifact: E:\Iran_stock_trade\financial-ml-decision-support\results\manifests\08_abstention_signal_policy.json
Policy manifest: E:\Iran_stock_trade\financial-ml-decision-support\results\manifests\08_abstention_policy_manifest.json
Policy configuration SHA: 4258a918e31d58a930b1ffc39520e5702f82bdd6f7be27288312ea83c8a474f4


## 8. Final integrity checks


In [9]:
assert policy_config["safeguards"][
    "unseen_test_loaded"
] is False
assert policy_config["safeguards"][
    "unseen_test_labels_used"
] is False
assert policy_config["safeguards"][
    "economic_returns_used_for_selection"
] is False
assert policy_config["safeguards"][
    "portfolio_metrics_used_for_selection"
] is False
assert selected_policy_artifact[
    "safeguards"
]["model_retrained"] is False
assert selected_policy_artifact[
    "safeguards"
]["minimum_signals_per_date_is_zero"] is True
assert selected_policy_artifact[
    "daily_cap"
]["zero_signal_dates_allowed"] is True
assert selected_policy_artifact[
    "coverage_constraints"
]["passed"] is True
assert selected_predictions["selected_signal"].sum() == int(
    selected_summary["signals"]
)
assert int(
    selected_predictions.loc[
        ~selected_predictions[
            "policy_eligible"
        ],
        "selected_signal",
    ].sum()
) == 0

print("Notebook 08 v3 integrity checks: PASSED")
print("Selected model:", SELECTED_MODEL)
print("Selected feature set:", SELECTED_FEATURE_SET)
print("Selected trial:", SELECTED_TRIAL)
print("Selected policy ID:", SELECTED_POLICY_ID)
print("Selected Breadth gate:", selected_gate_name)
print(
    "Allowed regimes:",
    list(selected_allowed_regimes),
)
print(
    "Minimum raw score:",
    selected_minimum_score,
)
print(
    "Threshold source quantile:",
    selected_threshold_quantile,
)
print(
    "Daily maximum fraction:",
    maximum_daily_fraction,
)
print("Minimum signals per date: 0")
print(
    "Baseline signals:",
    baseline_summary["signals"],
)
print(
    "Selected signals:",
    selected_summary["signals"],
)
print(
    "Signal coverage:",
    selected_summary["signal_coverage"],
)
print(
    "Zero-signal dates:",
    selected_summary["zero_signal_dates"],
)
print(
    "True positives:",
    selected_summary["true_positive"],
)
print(
    "False positives:",
    selected_summary["false_positive"],
)
print(
    "Precision:",
    selected_summary["precision"],
)
print(
    "Specificity:",
    selected_summary["specificity"],
)
print(
    "Sensitivity:",
    selected_summary["sensitivity"],
)
print(
    "Minimum fold coverage:",
    selected_summary[
        "minimum_fold_signal_coverage"
    ],
)
print("Unseen test loaded: False")
print("Economic returns used: False")
print("Portfolio metrics used: False")
print("Next stage: apply this exact frozen abstention policy to the historical post-hoc evaluation period.")


Notebook 08 v3 integrity checks: PASSED
Selected model: xgboost
Selected feature set: I_full_40
Selected trial: 10
Selected policy ID: G3_recovery_only__q0000
Selected Breadth gate: G3_recovery_only
Allowed regimes: ['recovery_negative', 'recovery_positive']
Minimum raw score: 0.3844631016254425
Threshold source quantile: 0.0
Daily maximum fraction: 0.05
Minimum signals per date: 0
Baseline signals: 3016
Selected signals: 1398
Signal coverage: 0.4635278514588859
Zero-signal dates: 452
True positives: 994
False positives: 404
Precision: 0.7110157367668097
Specificity: 0.9807774658609697
Sensitivity: 0.03224864549200272
Minimum fold coverage: 0.29508196721311475
Unseen test loaded: False
Economic returns used: False
Portfolio metrics used: False
Next stage: apply this exact frozen abstention policy to the historical post-hoc evaluation period.


## Review before freezing

Inspect all `08_abstention_` audits and both new manifest files. Do not alter
the selected gate or threshold after viewing the 2021–2024 results. The next
stage is a post-hoc diagnostic application, not independent confirmation.
